# Practice 2: Training

## Train ResNet18 and DenseNet121 on CIFAR-10 with TensorBoard logging

Train all four model variants (ResNet18 frozen/finetune, DenseNet121 frozen/finetune) on CIFAR-10.  Log loss and accuracy to TensorBoard.  Save best checkpoints for evaluation.

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Setup | Imports, device, data loaders, models | `src/data/load_cifar10` + `src/models/build_model` |
| 2 | Frozen: ResNet18 | Train frozen backbone, lr=1e-3, bs=64 | `src/training/train_model` |
| 3 | Frozen: DenseNet121 | Train frozen backbone, lr=1e-3, bs=64 | `src/training/train_model` |
| 4 | Fine-tune: ResNet18 | Train last block + FC, lr=1e-4, bs=64 | `src/training/train_model` |
| 5 | Fine-tune: DenseNet121 | Train last block + classifier, lr=1e-4, bs=64 | `src/training/train_model` |
| 6 | Loss/accuracy curves | Side-by-side comparison plots | `matplotlib` |
| 7 | Launch TensorBoard | Serve logs for interactive review | `tensorboard` |

---


## 1. Setup


In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

from src.data.load_cifar10 import get_cifar10_loaders
from src.models.build_model import (
    build_densenet121,
    count_trainable_params, count_all_params,
)
from src.training.train_model import train_model

# Device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Data
NUM_EPOCHS = 10
BATCH_SIZE = 64
LR_FROZEN = 1e-3
LR_FINETUNE = 1e-4
WEIGHT_DECAY = 1e-4

train_loader, val_loader, test_loader = get_cifar10_loaders(
    batch_size=BATCH_SIZE, num_workers=2
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Build all four model variants
rn_frozen  = build_resnet18(num_classes=10, mode="frozen", device=device)
dn_frozen  = build_densenet121(num_classes=10, mode="frozen", device=device)
rn_finetune = build_resnet18(num_classes=10, mode="finetune", device=device)
dn_finetune = build_densenet121(num_classes=10, mode="finetune", device=device)

criterion = nn.CrossEntropyLoss()

# Training run configs: (name, model, lr, mode)
RUN_CONFIGS = [
    ('ResNet18-frozen',     rn_frozen,     LR_FROZEN,   "frozen"),
    ('DenseNet121-frozen',  dn_frozen,     LR_FROZEN,   "frozen"),
    ('ResNet18-finetune',   rn_finetune,   LR_FINETUNE, "finetune"),
    ('DenseNet121-finetune',dn_finetune,   LR_FINETUNE, "finetune"),
]

results = {}
print("\u2713 Setup complete")


## 2. Train All Variants

Each run uses a separate TensorBoard writer with a unique tag. Best checkpoints are saved to ``experiments/checkpoints/``.

For **frozen** variants, the final classifier trains at the full LR.  For **fine-tune** variants, the last block trains at ``0.1 * LR`` and the final classifier at the full LR (discriminative fine-tuning).


In [ ]:
TB_LOG_DIR = "experiments/tb_logs"
CKPT_DIR = "experiments/checkpoints"
os.makedirs(TB_LOG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

for run_name, model, lr, mode in RUN_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"  Mode: {mode}, LR: {lr}, Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}")
    print(f"  Trainable params: {count_trainable_params(model):,} / {count_all_params(model):,}")
    print(f"{'='*60}")

    # Create optimizer with per-param-group LR for fine-tune variants
    if mode == "finetune":
        backbone_params = []
        head_params = []
        for name, param in model.named_parameters():
            if param.requires_grad:
                if "classifier" in name or name == "fc.weight" or name == "fc.bias":
                    head_params.append(param)
                else:
                    backbone_params.append(param)
        optimizer = torch.optim.AdamW([
            {"params": backbone_params, "lr": lr * 0.1},
            {"params": head_params, "lr": lr},
        ], weight_decay=WEIGHT_DECAY)
    else:
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=WEIGHT_DECAY,
        )

    # TensorBoard writer for this run
    writer = SummaryWriter(log_dir=os.path.join(TB_LOG_DIR, run_name))

    result = train_model(
        model, train_loader, val_loader, criterion, optimizer,
        device, num_epochs=NUM_EPOCHS,
        run_name=run_name, writer=writer,
        save_dir=CKPT_DIR,
    )

    writer.close()
    results[run_name] = result

print("\nAll training runs complete.")


## 3. Loss and Accuracy Curves

Plot training and validation loss/accuracy for all four runs side by side.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {
    "ResNet18-frozen": "#1f77b4",
    "DenseNet121-frozen": "#ff7f0e",
    "ResNet18-finetune": "#2ca02c",
    "DenseNet121-finetune": "#d62728",
}

# Loss plot
ax = axes[0, 0]
for name, res in results.items():
    epochs = range(1, len(res["train_losses"]) + 1)
    ax.plot(epochs, res["train_losses"], color=colors[name], linestyle="-", label=f"{name} (train)")
    ax.plot(epochs, res["val_losses"], color=colors[name], linestyle="--", label=f"{name} (val)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Accuracy plot
ax = axes[0, 1]
for name, res in results.items():
    epochs = range(1, len(res["train_accs"]) + 1)
    ax.plot(epochs, res["train_accs"], color=colors[name], linestyle="-", label=f"{name} (train)")
    ax.plot(epochs, res["val_accs"], color=colors[name], linestyle="--", label=f"{name} (val)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Training and Validation Accuracy")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Train accuracy zoom
ax = axes[1, 0]
for name, res in results.items():
    epochs = range(1, len(res["train_accs"]) + 1)
    ax.plot(epochs, res["train_accs"], color=colors[name], marker="o", markersize=4, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train Accuracy (%)")
ax.set_title("Training Accuracy (per model)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Val accuracy zoom
ax = axes[1, 1]
for name, res in results.items():
    epochs = range(1, len(res["val_accs"]) + 1)
    ax.plot(epochs, res["val_accs"], color=colors[name], marker="s", markersize=4, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val Accuracy (%)")
ax.set_title("Validation Accuracy (per model)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Results Summary


In [ ]:
print(f"{'Model':25s} {'Mode':10s} {'Best Val Loss':15s} {'Best Val Acc':15s} {'Best Epoch':12s}")
print("-" * 80)
for name, res in results.items():
    mode = "frozen" if "frozen" in name else "finetune"
    best_idx = res["best_epoch"]
    best_val_loss = res["val_losses"][best_idx - 1] if best_idx > 0 else "?"
    best_val_acc = res["val_accs"][best_idx - 1] if best_idx > 0 else "?"
    print(f"{name:25s} {mode:10s} {best_val_loss:<15.4f} {best_val_acc:<15.2f} {best_idx:<12d}")


## 5. Launch TensorBoard

To view all runs interactively, run in terminal:
```
tensorboard --logdir experiments/tb_logs
```

Then open the URL (default ``http://localhost:6006``) to compare loss/accuracy curves across all four variants.
